# KnowBase AI - experiments

In [5]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\MONEEM\NIAI Internship\KnowBase_AI
['.env', '.git', '.gitignore', '.ipynb_checkpoints', '.vscode', 'assets', 'chatbot.ipynb', 'requirements.txt', 'venv']


In [41]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv
api_key = os.getenv("GROQ_API_KEY")
print("Key loaded:", api_key is not None)

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

Key loaded: True


In [42]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain RAG in two sentences."}],
)
print(response.choices[0].message.content)

Retrieval‑Augmented Generation (RAG) is an architecture that combines a large language model with an external knowledge base: the model first retrieves relevant documents (or passages) from the database and then conditions its generation on that retrieved information. This approach lets the system produce up‑to‑date, factually grounded answers while still leveraging the fluent language capabilities of the generative model.


# Extract PDF text

In [43]:
import fitz #PyMuPDF

PDF_PATH = "Moneem Resume.pdf"

doc = fitz.open("C:/Users/MONEEM/NIAI Internship/KnowBase_AI/assets/Muhammad Moneem Resume.pdf")
pages = [(i + 1, page.get_text().strip()) for i, page in enumerate(doc)]
pages = [(num, text) for num, text in pages if text]  # drop empty pages

print("Total pages:", len(doc))
print("Pages with text:", len(pages))
print("\n--- Page", pages[0][0], "preview ---")
print(pages[0][1][:500])

Total pages: 1
Pages with text: 1

--- Page 1 preview ---
Muhammad Moneem Raza 
moneemraxa@gmail.com   |   +923081125201   |   Lahore, Pakistan   |   LinkedIn   |   GitHub   |   Portfolio 
Summary 
Full-stack software developer specializing in Flutter, with a track record of building scalable, cross-platform mobile applications 
backed by Firebase and REST APIs. Experienced in delivering features for a digital banking application, with a focus on clean 
architecture, performance, and reliable user experiences. BS Computer Science graduate, now expandin


# Chunking

In [44]:
CHUNK_SIZE = 900   # characters per chunk
OVERLAP = 150 # characters shared with the next chunk

def chunk_pages(pages, source):
    chunks = []
    step = CHUNK_SIZE - OVERLAP
    for page_no, text in pages:
        for start in range(0, len(text), step):
            piece = text[start:start + CHUNK_SIZE].strip()
            if len(piece) > 50:
                chunks.append({"text": piece, "source": source, "page": page_no})
    return chunks

chunks = chunk_pages(pages, PDF_PATH)

print("Total chunks:", len(chunks))

# Add this check to prevent the IndexError crash
if len(chunks) > 0:
    print("\nChunk 0:", chunks[0]["source"], "page", chunks[0]["page"])
    print(chunks[0]["text"][:300])
    if len(chunks) > 1:
        print("\nChunk 1 starts with:", chunks[1]["text"][:100])
else:
    print("\n⚠️ No chunks were created. Check if 'pages' contains any text!")
    print("Current 'pages' content:", pages)

Total chunks: 4

Chunk 0: Moneem Resume.pdf page 1
Muhammad Moneem Raza 
moneemraxa@gmail.com   |   +923081125201   |   Lahore, Pakistan   |   LinkedIn   |   GitHub   |   Portfolio 
Summary 
Full-stack software developer specializing in Flutter, with a track record of building scalable, cross-platform mobile applications 
backed by Firebase and REST

Chunk 1 starts with: elancing at Innovista learneasy.pk 
Skills 
Mobile Development: Flutter, Firebase, Firestore 
Tools:


# Embeddings and FAISS

In [45]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

vectors = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
).astype("float32")

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Vector shape:", vectors.shape)
print("Vectors in index:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector shape: (4, 384)
Vectors in index: 4


In [46]:
#retrival function and test

TOP_K = 4
MIN_SCORE = 0.25

def retrieve(question, k=TOP_K, min_score=MIN_SCORE):
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, min(k, len(chunks)))
    return [
        {**chunks[i], "score": float(s)}
        for s, i in zip(scores[0], ids[0])
        if i != -1 and s >= min_score
    ]

# Change this to a question your PDF can actually answer
question = "What is this document about?"

for hit in retrieve(question):
    print(f"Page {hit['Page']} | score {hit['score']:.2f}")
    print(hit["text"][:200], "\n")

# RAG answers with sources

In [47]:
import re

RAG_PROMPT = (
    "You are KnowBase AI. Answer the user's question using ONLY the document "
    "excerpts provided below. Each excerpt is labelled with its source and page. "
    "If the excerpts do not contain enough information to answer, reply exactly: "
    "\"I couldn't find that in the uploaded document.\" "
    "Do not invent facts. Mention page numbers when helpful."
)

history = [] # conversation memory

def clean(text):
    # Some models (e.g. Qwen) print their reasoning inside <think> tags; remove it
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def ask(question):
    hits = retrieve(question)

    if hits:
        context = "\n\n".join(
            f"[Source: {h['source']}, page {h['page']}]\n{h['text']}" for h in hits
        )

    else:
        context = "(no relevant excrepts found)"

    messages = (
        [{"role": "system", "content": f"{RAG_PROMPT}\n\nDOCUMENT EXCERPTS:\n{context}"}]
        + history[-8:]
        + [{"role": "user", "content": question}]
    )

    response = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.2
    )
    answer = clean(response.choices[0].message.content)

    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})

    print("Answer:\n", answer)
    print("\nSources:")
    for h in hits:
        print(f" - {h['source']}, page {h['page']} (score {h['score']:.2f})")


In [55]:
try:
    ask("Summarize the main points of this document.")
except Exception as e:
    print(e)

Answer:
 I couldn't find that in the uploaded document.

Sources:


In [49]:
# Diagnostic cell
q = embedder.encode(["Summarize the main points of this document."], normalize_embeddings=True).astype("float32")
scores, ids = index.search(q, 5)
for s, i in zip(scores[0], ids[0]):
    print(f"score {s:.2f} | page {chunks[i]['page']} | {chunks[i]['text'][:80]!r}")

score 0.14 | page 1 | 'Muhammad Moneem Raza \nmoneemraxa@gmail.com   |   +923081125201   |   Lahore, Pak'
score 0.14 | page 1 | 'elancing at Innovista learneasy.pk \nSkills \nMobile Development: Flutter, Firebas'
score 0.12 | page 1 | 'le and scalable \nsolution. \nCravely – Restaurant Application \n\uf0b7 \nDesigned and de'
score 0.09 | page 1 | ', and Firebase Cloud Messaging (FCM). \n\uf0b7 \nBuilt reusable UI components, followed'
score -340282346638528859811704183484516925440.00 | page 1 | 'le and scalable \nsolution. \nCravely – Restaurant Application \n\uf0b7 \nDesigned and de'


In [54]:
history.clear()
ask("Which programming languages and frameworks does Moneem know?")
ask("What is the Cravely project about?")
ask("What is the capital of Mars?")

Answer:
 **Programming languages**

- Dart  
- SQL  
- JavaScript  
- C#  
- C++  
- Python  
- HTML  
- CSS  

**Frameworks / platforms**

- Flutter (mobile development framework)  
- Firebase (including Cloud Firestore)  
- REST APIs (used for backend integration)  

*Source: Moneem’s resume, page 1.*

Sources:
 - Moneem Resume.pdf, page 1 (score 0.30)
 - Moneem Resume.pdf, page 1 (score 0.27)
Answer:
 The **Cravely** project is a **cross‑platform restaurant application**.  
Moneem designed and built it using **Flutter** for the UI, **Firebase** (including Authentication, Google Sign‑In, Cloud Firestore, and Cloud Messaging) and **REST APIs** for backend communication. The app follows a clean, modular architecture with reusable components, uses Provider for state management, provides a responsive UI, and its APIs were tested with Postman.  

*Source: Moneem Resume, page 1.*

Sources:
 - Moneem Resume.pdf, page 1 (score 0.28)
 - Moneem Resume.pdf, page 1 (score 0.25)
Answer:
 I couldn